# Diplomado de Grado en Análisis de Datos Aplicado a la Toma de Decisiones
## UNICOLOMBO — Fundación Universitaria Colombo Internacional · Cartagena
**Módulo 2:** Preparación y Análisis Exploratorio de Datos  
**Sesión 3:** Transformación de Datos  
**Docente:** Ing. Heyder Medrano Olier | **Período:** 2026 — Vacaciones

---
**Tipo:** Notebook de Actividad  
**Entrega:** 1 de julio de 2026 · 23:59 (hora Colombia)  
**Valor:** 5.0 puntos  

**Estudiante:** Jefferson Arrieta Duran

> Aplique las técnicas de transformación al dataset DataRetail LATAM.
> Justifique cada elección de técnica con celdas Markdown.

## 1. Librerías

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, warnings
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                    LabelEncoder, OrdinalEncoder, OneHotEncoder)
from sklearn.model_selection import train_test_split
import joblib
warnings.filterwarnings('ignore')
print('OK')

## 2. Cargar Dataset Limpio

In [ ]:
import numpy as np, pandas as pd, random
from datetime import datetime, timedelta
np.random.seed(42); random.seed(42)
N=2000
ciudades=["Bogotá","Medellín","Cali","Barranquilla","Cartagena",
          "Lima","Ciudad de México","Buenos Aires","Santiago","Montevideo"]
paises=["Colombia","Perú","México","Argentina","Chile","Uruguay"]
cats=["Computación","Periféricos","Audio","Telefonía","Accesorios","Gaming","Componentes"]
canales=["Tienda Física","E-commerce","Distribuidor","Corporativo","Marketplace"]
prods=["Laptop Pro","Tablet Ultra","Monitor 4K","Teclado Inalámbrico",
       "Auriculares BT","Parlante Portátil","Smartphone Galaxy",
       "Cámara Web HD","Smartwatch Pro","Mouse Gaming","SSD 1TB","GPU RTX 4060"]
precios={"Laptop Pro":1200,"Tablet Ultra":450,"Monitor 4K":380,"Teclado Inalámbrico":75,
         "Auriculares BT":120,"Parlante Portátil":90,"Smartphone Galaxy":680,
         "Cámara Web HD":95,"Smartwatch Pro":250,"Mouse Gaming":65,"SSD 1TB":130,"GPU RTX 4060":580}
rows=[]
for i in range(N):
    prod=random.choice(prods); cat=cats[prods.index(prod)%len(cats)]
    pais=random.choice(paises); ciudad=random.choice(ciudades); canal=random.choice(canales)
    qty=random.randint(1,20); precio=round(precios[prod]*np.random.uniform(0.8,1.3),2)
    desc=round(random.choice([0,0,0,0.05,0.10,0.15,0.20,0.25,0.30]),2)
    total=round(qty*precio*(1-desc),2); margen=round(total*np.random.uniform(0.05,0.35),2)
    fecha=datetime(2024,1,1)+timedelta(days=random.randint(0,729))
    rows.append({"id_venta":f"V{i+1:05d}","fecha_venta":fecha,"id_cliente":f"C{random.randint(1,500):04d}",
                 "nombre_cliente":f"Cliente {random.randint(1,500)}",
                 "ciudad":ciudad,"pais":pais,"id_producto":f"P{prods.index(prod)+1:02d}",
                 "nombre_producto":prod,"categoria":cat,"canal_venta":canal,
                 "cantidad":qty,"precio_unitario":precio,"descuento":desc,
                 "total_venta":total,"margen_utilidad":margen})
df = pd.DataFrame(rows)
# Este dataset ya está limpio (post-S02)
print(f"Dataset limpio S02→S03: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Nulos: {df.isna().sum().sum()} | Tipos: OK")
df.head(20)


## 3. Feature Engineering — Variables Temporales

Descomponga `fecha_venta` en componentes útiles para el análisis.

In [ ]:
df['anio']          = df['fecha_venta'].dt.year
df['mes']           = df['fecha_venta'].dt.month
df['trimestre']     = df['fecha_venta'].dt.quarter
df['dia_semana']    = df['fecha_venta'].dt.dayofweek
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)
print('Features temporales creadas:')
print(df[['anio','mes','trimestre','dia_semana','es_fin_semana']].head(5).to_string())

### 📝 ¿Por qué descomponer fechas?
> [
> Al mantener **`fecha_venta`** únicamente como una variable de tipo *datetime*, se desaprovecha información importante que puede influir en el comportamiento de las ventas. Al descomponerla en componentes como **año, mes, trimestre, día de la semana** y si la venta ocurrió en **fin de semana**, es posible identificar patrones temporales, estacionalidades y tendencias de compra. Estas nuevas variables facilitan el análisis exploratorio y permiten que los modelos predictivos aprovechen mejor la información relacionada con el tiempo, ya que la fecha completa por sí sola no refleja de manera explícita estas características.
]

> Explique qué información pierde al dejar fecha_venta como datetime crudo
Mantener fecha_venta únicamente como una variable de tipo datetime limita el análisis de la información temporal contenida en ella. De esta manera, resulta más difícil estudiar el comportamiento de las ventas según el año, el mes, el trimestre o el día de la semana, así como identificar diferencias entre días hábiles y fines de semana. Al separar la fecha en estos componentes, es posible detectar tendencias, reconocer patrones estacionales y realizar comparaciones entre distintos periodos, lo que contribuye a un análisis más detallado y a una mejor toma de decisiones basada en la evolución de las ventas.

## 4. Feature Engineering — Variables de Negocio

In [ ]:
df['margen_pct']       = (df['margen_utilidad']/df['total_venta'].replace(0,np.nan)).fillna(0).round(4)
df['precio_neto']      = (df['precio_unitario']*(1-df['descuento'])).round(2)
df['total_x_cliente']  = df.groupby('id_cliente')['total_venta'].transform('sum').round(2)
df['frecuencia_cliente']= df.groupby('id_cliente')['id_venta'].transform('count')
df['segmento_precio']  = pd.cut(df['precio_unitario'],bins=[0,100,400,2000],
                                 labels=['Económico','Medio','Premium'])
print('Features de negocio:', ['margen_pct','precio_neto','total_x_cliente','frecuencia_cliente','segmento_precio'])

Justificación

La incorporación de estas variables de negocio tiene como propósito ampliar la información disponible en el conjunto de datos, permitiendo realizar un análisis más completo del desempeño de las ventas y del comportamiento de los clientes. Estas nuevas características aportan indicadores que facilitan la evaluación de la rentabilidad, la identificación de patrones de compra y la clasificación de los productos, proporcionando información útil para apoyar la toma de decisiones comerciales.

Descripción de las variables

margen_pct: expresa el porcentaje de ganancia obtenido en relación con el valor total de la venta, permitiendo comparar el nivel de rentabilidad entre diferentes transacciones, independientemente de su monto.
precio_neto: corresponde al precio final del producto una vez aplicado el descuento, lo que facilita medir el efecto de las estrategias promocionales sobre las ventas.
total_x_cliente: refleja el valor acumulado de las compras realizadas por cada cliente, permitiendo reconocer aquellos que generan un mayor aporte a los ingresos de la empresa.
frecuencia_cliente: registra la cantidad de compras efectuadas por cada cliente, siendo un indicador útil para analizar sus hábitos de compra y su nivel de fidelidad.
segmento_precio: organiza los productos en las categorías Económico, Medio y Premium, lo que facilita el estudio del comportamiento de las ventas según el rango de precios y el diseño de estrategias comerciales dirigidas a cada segmento.

In [ ]:
import matplotlib.pyplot as plt

variables = ['precio_unitario','total_venta','margen_utilidad']

for col in variables:
    plt.figure(figsize=(6,4))
    plt.hist(df[col], bins=30)
    plt.title(f'{col} - Antes de escalar')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

# RobustScaler
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

df_scaled = df.copy()

df_scaled[variables] = scaler.fit_transform(df[variables])

for col in variables:
    plt.figure(figsize=(6,4))
    plt.hist(df_scaled[col], bins=30)
    plt.title(f'{col} - Después de escalar')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.show()

Justificación

Se eligió RobustScaler debido a la presencia de valores atípicos en variables como precio_unitario y total_venta. A diferencia de otros métodos de escalado, este utiliza la mediana y el rango intercuartílico (IQR) como referencia, lo que disminuye el impacto de los valores extremos sobre los datos. Gracias a ello, se obtiene un conjunto de variables escaladas de forma más consistente, favoreciendo tanto el análisis de la información como el desempeño de futuros modelos predictivos.

📝 Interpretación de margen_pct

La variable margen_pct permite conocer la rentabilidad de cada venta en términos porcentuales, mostrando qué proporción del ingreso obtenido corresponde a utilidad. Por su parte, margen_utilidad únicamente refleja el valor monetario de la ganancia, sin considerar el tamaño de la venta. Al trabajar con margen_pct, es posible comparar el nivel de rentabilidad entre transacciones de diferentes valores, identificar aquellas que generan un mejor rendimiento y obtener una visión más precisa de la eficiencia comercial de cada venta.

## 5. One-Hot Encoding — canal_venta

In [ ]:
df_ohe = pd.get_dummies(df, columns=['canal_venta'], prefix='canal', drop_first=False)
cols_canal = [c for c in df_ohe.columns if c.startswith('canal_')]
print('Columnas canal:', cols_canal)
print(df_ohe[cols_canal].head(5).to_string())

📝 ¿Por qué no utilizar Label Encoding para canal_venta?

No es conveniente aplicar Label Encoding a la variable canal_venta porque se trata de una variable categórica nominal, es decir, sus categorías no siguen ningún tipo de orden o jerarquía. Al asignarles valores numéricos consecutivos, algunos algoritmos de aprendizaje automático podrían asumir que existe una relación de mayor o menor importancia entre los distintos canales, cuando realmente no es así.

En estos casos, la alternativa más adecuada es One-Hot Encoding, ya que transforma cada categoría en una variable binaria independiente (0 o 1). De esta forma, cada canal se representa sin establecer relaciones de orden que puedan afectar el análisis o el rendimiento de los modelos predictivos.

Ejemplo:

Si se aplicara Label Encoding, la codificación podría quedar así:

CORPORATIVO = 0
DISTRIBUIDOR = 1
E-COMMERCE = 2
MARKETPLACE = 3
TIENDA FÍSICA = 4

Con esta representación, un modelo podría interpretar equivocadamente que TIENDA FÍSICA (4) tiene un valor o prioridad superior a CORPORATIVO (0), simplemente por el número asignado. Sin embargo, los canales de venta son categorías independientes y no existe un orden entre ellas. Por esta razón, One-Hot Encoding es la técnica más apropiada para codificar esta variable.

##One-Hot Encoding para ciudad y categoría

In [ ]:
print("Columnas antes:", df.shape[1])

df_ohe = pd.get_dummies(
    df,
    columns=['ciudad','categoria','canal_venta'],
    drop_first=False
)

print("Columnas después:", df_ohe.shape[1])

Justificación

Se empleó One-Hot Encoding para transformar las variables ciudad, categoría y canal_venta, ya que corresponden a variables categóricas nominales cuyas categorías no presentan una secuencia ni una jerarquía definida. Este método convierte cada categoría en una variable binaria independiente, evitando que los modelos interpreten relaciones de orden inexistentes y permitiendo una representación más adecuada de la información para el análisis y la construcción de modelos predictivos.

## 6. Ordinal Encoding — segmento_precio

In [ ]:
oe = OrdinalEncoder(categories=[['Económico','Medio','Premium']])
df['segmento_precio_num'] = oe.fit_transform(df[['segmento_precio']])
print(df[['segmento_precio','segmento_precio_num']].drop_duplicates().sort_values('segmento_precio_num'))

📝 ¿Por qué utilizar Ordinal Encoding en este caso?

Se aplicó Ordinal Encoding a la variable segmento_precio porque sus categorías presentan una secuencia lógica basada en el nivel de precio de los productos. En este caso, existe una relación jerárquica entre los segmentos, donde Económico representa el nivel más bajo, seguido de Medio y finalmente Premium como el nivel más alto. Al asignar valores numéricos consecutivos, esta relación de orden se conserva y puede ser aprovechada tanto en el análisis de datos como en los modelos de aprendizaje automático.

A diferencia de variables como canal_venta, cuyas categorías son independientes entre sí, segmento_precio sí tiene un significado asociado al orden de sus niveles, por lo que una codificación ordinal resulta apropiada.

Ejemplo:

Económico → 0
Medio → 1
Premium → 2

Con esta codificación, los modelos pueden reconocer que un producto del segmento Premium pertenece a un nivel de precio superior al Medio, y este, a su vez, está por encima del Económico. Por ello, Ordinal Encoding es la técnica más adecuada para representar esta variable.

In [ ]:
df['revenue_per_unit'] = (
    df['total_venta']/
    df['cantidad']
).round(2)

In [ ]:
df['es_descuento_alto'] = (
    df['descuento']>0.15
).astype(int)

Justificación

La creación de estas variables tiene como objetivo complementar el análisis de las ventas mediante indicadores que aportan información adicional sobre el desempeño comercial. Por un lado, permiten conocer el ingreso obtenido por cada unidad vendida y, por otro, facilitan la identificación de aquellas transacciones en las que se aplicaron descuentos elevados. Esta información resulta útil para evaluar el impacto de las estrategias promocionales y comprender mejor el comportamiento de las ventas.

## 7. Escalado — Comparación de 3 Scalers

In [ ]:
cols_n = ['precio_unitario','total_venta','cantidad','margen_utilidad']
X = df[cols_n].fillna(0)

ss,mms,rs = StandardScaler(), MinMaxScaler(), RobustScaler()
Xs = ss.fit_transform(X);  Xm = mms.fit_transform(X);  Xr = rs.fit_transform(X)

print('Estadísticas precio_unitario después de escalar:')
i = cols_n.index('precio_unitario')
print(f'  Standard: media={Xs[:,i].mean():.4f}  std={Xs[:,i].std():.4f}')
print(f'  MinMax:   min={Xm[:,i].min():.4f}    max={Xm[:,i].max():.4f}')
print(f'  Robust:   media={Xr[:,i].mean():.4f}  std={Xr[:,i].std():.4f}')

📝 ¿Qué Scaler elegiría para DataRetail y por qué?

Para el conjunto de datos DataRetail LATAM, la alternativa más apropiada es RobustScaler, debido a que durante el análisis exploratorio se detectó la presencia de valores atípicos, específicamente 160 registros en total_venta y 96 en precio_unitario. Aunque estos casos fueron mitigados mediante la técnica de Winsorizing (IQR Cap), aún es posible que algunos valores extremos continúen influyendo en la distribución de los datos.

La principal ventaja de RobustScaler es que realiza el escalado utilizando la mediana y el rango intercuartílico (IQR), en lugar de la media y la desviación estándar. Esto hace que su desempeño sea mucho más resistente frente a la presencia de observaciones atípicas, obteniendo una transformación más consistente de las variables numéricas.

En contraste, StandardScaler basa su cálculo en la media y la desviación estándar, por lo que los valores extremos pueden alterar significativamente el resultado del escalado. Por otro lado, MinMaxScaler normaliza los datos dentro del intervalo [0,1]; sin embargo, cuando existen outliers, la mayoría de las observaciones tienden a concentrarse en un rango muy reducido, disminuyendo la capacidad para distinguir diferencias entre ellas.

Por estas razones, RobustScaler representa la mejor elección para este conjunto de datos, ya que reduce el efecto de los valores atípicos, conserva de mejor manera la distribución de las variables y proporciona datos más adecuados para el desarrollo de modelos de análisis y aprendizaje automático orientados a apoyar la toma de decisiones en DataRetail LATAM.

## 8. Train/Test Split Correcto

In [ ]:
X_feat = df[['cantidad','precio_unitario','descuento','total_venta',
              'margen_utilidad','anio','mes','trimestre','dia_semana']].fillna(0)
y = df['margen_pct']
X_train, X_test, y_train, y_test = train_test_split(X_feat, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
# CORRECTO: fit solo en train
scaler = RobustScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)  # transform, NO fit_transform
print('Scaler aplicado correctamente (fit solo en train)')

📝 ¿Qué es Data Leakage?

El Data Leakage ocurre cuando, durante el proceso de entrenamiento de un modelo, se utiliza información que pertenece al conjunto de prueba y que, en condiciones reales, no estaría disponible. Como consecuencia, el modelo obtiene un conocimiento indebido sobre esos datos, lo que produce resultados de evaluación más altos de lo que realmente serían al enfrentarse a información nueva.

Si se ejecutara fit_transform() sobre X_test, el método calcularía los parámetros del escalado (como la mediana y el rango intercuartílico en el caso de RobustScaler, o la media y la desviación estándar en otros escaladores) utilizando los datos del conjunto de prueba. Esto implica que la información de X_test influiría en el proceso de transformación, generando una evaluación poco objetiva y afectando la capacidad de medir el verdadero rendimiento del modelo.

Para evitar este problema, la práctica correcta consiste en aplicar fit_transform() únicamente sobre X_train, permitiendo que el escalador aprenda sus parámetros exclusivamente con los datos de entrenamiento. Posteriormente, sobre X_test solo debe utilizarse transform(), de manera que se aplique la misma transformación sin volver a calcular los parámetros. Así se evita el Data Leakage y se obtiene una estimación más confiable del desempeño del modelo frente a datos no vistos.

## 9. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Features Transformadas — S03', fontweight='bold')
df.groupby('mes')['total_venta'].mean().plot(ax=axes[0], marker='o')
axes[0].set_title('Ventas por mes')
df['segmento_precio'].value_counts().plot(kind='bar', ax=axes[1], ec='white')
axes[1].set_title('Distribución segmento_precio'); axes[1].tick_params(axis='x', rotation=0)
df['margen_pct'].hist(ax=axes[2], bins=40, ec='white')
axes[2].set_title('Distribución margen_pct')
plt.tight_layout(); plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.select_dtypes(include='number').corr()

plt.figure(figsize=(12,8))
sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)
plt.title("Matriz de correlación")
plt.show()

In [ ]:
corr['total_venta'].sort_values(ascending=False)

##Interpretación

El análisis de correlación permitió identificar que las variables con mayor asociación con total_venta son margen_utilidad (0.884460), revenue_per_unit (0.775508) y precio_neto (0.775508). Estos resultados evidencian que el monto total de las ventas está estrechamente relacionado con la utilidad obtenida, el ingreso promedio por cada unidad comercializada y el precio final del producto después de aplicar los descuentos. Debido a esta fuerte relación, estas variables representan factores importantes para comprender el comportamiento de las ventas y pueden contribuir a mejorar el desempeño de modelos predictivos y el análisis comercial en DataRetail LATAM.

¿Por qué estas variables?
margen_utilidad (0.884460): presenta la correlación más alta con total_venta, lo que indica que, en términos generales, las ventas con un mayor valor también generan una utilidad más elevada.
revenue_per_unit (0.775508): refleja el ingreso obtenido por cada unidad vendida, por lo que un incremento en este indicador suele traducirse en un mayor valor total de la venta.
precio_neto (0.775508): corresponde al precio del producto una vez aplicado el descuento. Al influir directamente en el valor que paga el cliente, mantiene una relación significativa con el monto final de cada transacción.

#PIPELINE

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler,OneHotEncoder

num_cols=[
'precio_unitario',
'total_venta',
'margen_utilidad'
]

cat_cols=[
'ciudad',
'categoria',
'canal_venta'
]

preprocessor=ColumnTransformer(
[
('num',RobustScaler(),num_cols),
('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
]
)

pipeline=Pipeline(
[
('preprocessor',preprocessor)
]
)

X=df.drop(columns=['margen_pct'])

pipeline.fit(X)

X_final=pipeline.transform(X)

print("Pipeline ejecutado correctamente.")

Justificación

La implementación de un Pipeline permite integrar todas las etapas de transformación de los datos en un único proceso, asegurando que se apliquen de forma consistente y en la secuencia correcta. Esto facilita la reproducibilidad del flujo de trabajo, reduce la posibilidad de cometer errores durante el preprocesamiento y evita que información del conjunto de prueba influya en el entrenamiento del modelo, minimizando así el riesgo de Data Leakage.

## 10. Guardar Transformadores

In [ ]:
joblib.dump(scaler, 'scaler_s03.pkl')
joblib.dump(oe,     'ordinal_encoder.pkl')
print('Guardados: scaler_s03.pkl, ordinal_encoder.pkl')
# Cómo cargar
# scaler_cargado = joblib.load('scaler_s03.pkl')
# X_nuevo = scaler_cargado.transform(X_produccion)

## 11. Exportar Dataset Transformado

In [ ]:
df.to_csv('DataRetail_LATAM_S03_transformado.csv', index=False, encoding='utf-8-sig')
print(f'Exportado: {df.shape[0]:,} filas x {df.shape[1]} columnas')


Como parte del proceso final, se exportó el conjunto de datos transformado con el propósito de contar con una versión preparada para análisis más avanzados y el desarrollo de modelos predictivos. Este archivo reúne todas las transformaciones y nuevas variables generadas durante la etapa de Feature Engineering, dejando un dataset enriquecido y listo para ser utilizado en futuras tareas de análisis y modelado.

## 12. Conclusiones

### Mis hallazgos

1. Impacto de las transformaciones en el negocio

Las transformaciones realizadas mejoran significativamente la calidad del conjunto de datos, permitiendo obtener información más detallada sobre el comportamiento de las ventas, los clientes y los productos. Esto facilita la detección de patrones, el análisis de la rentabilidad, la segmentación de clientes y el diseño de estrategias comerciales más efectivas. Además, el dataset queda preparado para el desarrollo de modelos predictivos que apoyen la toma de decisiones.

2. Variable creada con mayor aporte

Entre las nuevas variables generadas, revenue_per_unit resultó ser una de las más relevantes, ya que permite calcular el ingreso promedio obtenido por cada unidad vendida. Este indicador facilita la comparación del rendimiento de los productos sin depender únicamente del valor total de la venta. Adicionalmente, su alta correlación con total_venta demuestra que es una característica valiosa para futuros análisis y modelos de predicción.

3. Técnica que generó mayor valor

La etapa de Feature Engineering fue la que más enriqueció el dataset, debido a que permitió incorporar nuevas variables derivadas de la información original. Indicadores como precio_neto, margen_pct, total_x_cliente, frecuencia_cliente y revenue_per_unit ofrecen una visión más completa del desempeño comercial y aportan información que no estaba disponible de forma directa en los datos iniciales.

4. Aspecto más importante durante el preprocesamiento

Uno de los errores que debe evitarse es el Data Leakage, ya que ocurre cuando el modelo utiliza información del conjunto de prueba durante la fase de entrenamiento. Esta situación provoca que la evaluación del modelo sea poco realista y genere resultados más favorables de los que realmente tendría al enfrentarse a datos nuevos. Por ello, es indispensable que las transformaciones aprendan únicamente a partir de los datos de entrenamiento.

5. Escalador más adecuado para el conjunto de datos

Considerando las características de DataRetail LATAM, RobustScaler es la alternativa más conveniente para el escalado de las variables numéricas. Durante el análisis exploratorio se identificó la presencia de numerosos valores atípicos, especialmente en total_venta y precio_unitario. Al basarse en la mediana y el rango intercuartílico (IQR), este método reduce el efecto de los valores extremos y proporciona un escalado más estable que otras técnicas como StandardScaler o MinMaxScaler

## Referencias
1. Pedregosa, F. et al. (2011). Scikit-learn. *JMLR*, 12, 2825-2830.
2. McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly.
3. Géron, A. (2023). *Hands-On Machine Learning* (3.ª ed.). O'Reilly.
4. Zheng, A. & Casari, A. (2018). *Feature Engineering for Machine Learning*. O'Reilly.
5. Kuhn, M. & Johnson, K. (2019). *Feature Engineering and Selection*. CRC Press.

---
*Notebook Actividad S03 — UNICOLOMBO · 2026*